In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

c:\Users\Adham\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Adham\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
dataset_path = '../5.Data/propertyfinder.csv'
df = pd.read_csv(dataset_path)
print(f"Original shape: {df.shape}")
df.head(3)

Original shape: (39713, 53)


,listing_id,internal_id,category,listing_type,detail_url,property_type,offering_type,completion_status,title,price_egp,...,agent_is_super,agent_languages,broker_id,broker_name,broker_email,broker_phone,contact_phone,contact_whatsapp,contact_email,scraped_at
0,F7QB31CGWE509V2W7DF2GARB2C,56009081.0,buy,property,https://www.propertyfinder.eg/en/plp/buy/duple...,Duplex,Residential for Sale,completed,Garden Villa - Lake View Boutique - Prime Loca...,24500000.0,...,False,NaN,5758.0,Spade consultancy,mohmedg.sedik@gmail.com,1.001437e+09,2.012018e+11,2.022126e+10,pierre.Osama@spade-consultancy.com,2026-03-04T14:20:33.281007
1,K1JC3D6N57ED52N3VX1QQKHHXG,56247925.0,buy,property,https://www.propertyfinder.eg/en/plp/buy/apart...,Apartment,Residential for Sale,off_plan,For Sale: Finished Apartment+ ACs in Village West,5145000.0,...,False,NaN,492.0,Abrag Real Estate,snawara71@gmail.com,2.010012e+11,2.010072e+11,2.022126e+10,ranoushamer901@gmail.com,2026-03-04T14:20:33.281007
2,Q6GEB8T6PZTJGNNPWA5PX3JCWR,56253883.0,buy,property,https://www.propertyfinder.eg/en/plp/buy/apart...,Apartment,Residential for Sale,completed,UnderMarket Price for Apt. 250 RTM PrimeLocation,10800000.0,...,False,English | Arabic,5279.0,Premises,mahmouddessam@hotmail.com,2.010051e+11,2.010910e+11,2.022126e+10,ashrafsobhy99@hotmail.com,2026-03-04T14:20:33.281007


In [3]:
# Drop columns that are completely missing, completely uniform (1 unique value), 
# or not useful for recommendation (e.g., redundant text strings, agent info, etc)
cols_to_drop = [
    'internal_id', 'detail_url', 'video_url', 'reference', 'description', 
    'agent_id', 'agent_name', 'agent_email', 'broker_id', 'broker_name', 
    'broker_email', 'broker_phone', 'contact_phone', 'contact_whatsapp', 
    'contact_email', 'scraped_at', 'rera', 'agent_languages', 'title', 'images_count',
    # Single value columns (zero variance):
    'price_currency', 'area_unit', 'is_verified', 'is_new_construction', 
    'is_direct_from_dev', 'is_exclusive', 'agent_is_super',
    # Redundant / low variance / leak columns:
    'location_full', 'listing_type', 'listed_date'
]
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

# Set index to listing_id if available
if 'listing_id' in df.columns:
    df.set_index('listing_id', inplace=True)
print(f"Shape after dropping useless features: {df.shape}")

Shape after dropping useless features: (39713, 22)


In [4]:
# Clean specific numeric string columns
def clean_bedrooms(val):
    if pd.isna(val): return np.nan
    val = str(val).lower().strip()
    if val == 'studio': return 0
    if val == '7+': return 7
    try: return float(val)
    except: return np.nan

if 'bedrooms' in df.columns:
    df['bedrooms'] = df['bedrooms'].apply(clean_bedrooms)

def clean_bathrooms(val):
    if pd.isna(val): return np.nan
    val = str(val).lower().strip()
    if val == '7+': return 7
    try: return float(val)
    except: return np.nan

if 'bathrooms' in df.columns:
    df['bathrooms'] = df['bathrooms'].apply(clean_bathrooms)

if 'area_value' in df.columns:
    df['area_value'] = pd.to_numeric(df['area_value'], errors='coerce')
    
if 'price_egp' in df.columns:
    df['price_egp'] = pd.to_numeric(df['price_egp'], errors='coerce')

In [5]:
# Handle missing values
# 1. Numeric features (impute with median)
num_cols = ['price_egp', 'lat', 'lon', 'bedrooms', 'bathrooms', 'area_value']
num_imputer = SimpleImputer(strategy='median')
num_cols_exist = [c for c in num_cols if c in df.columns]
if num_cols_exist:
    df[num_cols_exist] = num_imputer.fit_transform(df[num_cols_exist])

# 2. Categorical features (fill missing with 'Unknown')
cat_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()
if 'amenities' in cat_cols:
    cat_cols.remove('amenities')

if cat_cols:
    df[cat_cols] = df[cat_cols].fillna('Unknown')

In [6]:
# Feature engineering
if 'amenities' in df.columns:
    df['amenities'] = df['amenities'].fillna('')
    df['amenities_count'] = df['amenities'].apply(lambda x: len(str(x).split('|')) if x else 0)
    # The string representation is dropped since the ML algorithm won't take flat strings
    df = df.drop(columns=['amenities'])


In [7]:
# Remove extreme outliers in price/area if applicable using IQR
def remove_outliers(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return data[(data[col] >= lower_bound) & (data[col] <= upper_bound)]

print(f"Dataframe size pre-outliers: {df.shape}")
if 'price_egp' in df.columns and 'area_value' in df.columns:
    df = remove_outliers(df, 'price_egp')
    df = remove_outliers(df, 'area_value')
print(f"Dataframe size post-outliers: {df.shape}")


Dataframe size pre-outliers: (39713, 22)
Dataframe size post-outliers: (34122, 22)


In [8]:
# Categorical Encoding & Scaling
le = LabelEncoder()
for col in cat_cols:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))
    
# Scale numeric features
scaler = StandardScaler()
if num_cols_exist:
    df[num_cols_exist] = scaler.fit_transform(df[num_cols_exist])

print("Preprocessing complete! Dataset is clean, outlier-filtered, and encoded.")
df.head(3)

Preprocessing complete! Dataset is clean, outlier-filtered, and encoded.


,category,property_type,offering_type,completion_status,price_egp,price_period,city,town,district,subdistrict,...,bedrooms,bathrooms,area_value,furnished,listing_level,is_premium,is_featured,has_view_360,payment_method,amenities_count
listing_id,,,,,,,,,,,,,,,,,,,,,
K1JC3D6N57ED52N3VX1QQKHHXG,0,0,1,3,0.173307,2,4,61,430,1014,...,-0.735487,-0.621510,-0.595724,1,2,1,0,0,1,11
Q6GEB8T6PZTJGNNPWA5PX3JCWR,0,0,1,1,1.178795,2,2,46,470,15,...,1.344345,1.366973,1.136283,2,2,1,0,0,1,18
PPH0852SAB266X6GSVATJP4F0M,0,0,1,1,1.125454,2,2,46,470,15,...,-0.735487,-0.621510,-0.418432,3,2,1,0,0,1,18


In [15]:
output_path = r"C:\Users\Adham\Desktop\AI_System\2.Recommender\propertyfinder_cleaned.csv"
df.to_csv(output_path)
print(f"Cleaned dataset saved to {output_path}")

Cleaned dataset saved to C:\Users\Adham\Desktop\AI_System\2.Recommender\propertyfinder_cleaned.csv
